# Product search with `HelixIndex`

In [11]:
%%capture
!pip install datasets sentence-transformers simlar simlar-engine ipywidgets

## Load the catalog

We use [`ashraq/fashion-product-images-small`](https://huggingface.co/datasets/ashraq/fashion-product-images-small)
— ~44,000 catalog products. Each row has a
`productDisplayName` (the searchable product text) plus structured attributes
(`gender`, `articleType`, `baseColour`, ...) we keep for display.

In [12]:
from datasets import load_dataset

ds = load_dataset("ashraq/fashion-product-images-small", split="train")

CORPUS = [str(name) for name in ds["productDisplayName"]]
CORPUS_SIZE = len(CORPUS)
IDS = [f"prod_{i}" for i in range(CORPUS_SIZE)] # unique string id per doc

id_to_text = dict(zip(IDS, CORPUS))
id_to_meta = {
    IDS[i]: {"type": ds["articleType"][i], "colour": ds["baseColour"][i], "gender": ds["gender"][i]}
    for i in range(CORPUS_SIZE)
}

In [13]:
print(f"Loaded {CORPUS_SIZE} products")
print("Preview:")
for name in CORPUS[:5]:
    print(f" - {name}")

Loaded 44072 products
Preview:
 - Turtle Check Men Navy Blue Shirt
 - Peter England Men Party Blue Jeans
 - Titan Women Silver Watch
 - Manchester United Men Solid Black Track Pants
 - Puma Men Grey T-shirt


## Embed the catalog

In [14]:
import numpy as np
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")
vectors = model.encode(CORPUS, normalize_embeddings=True, show_progress_bar=True).astype(np.float32)

print(f"Catalog vectors: {vectors.shape} dtype: {vectors.dtype}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/1378 [00:00<?, ?it/s]

Catalog vectors: (44072, 384) dtype: float32


## Using simlar (hybrid index)

In [15]:
%%time
from simlar import HelixIndex

index = HelixIndex(top_k=20, text_k=100, vector_k=100)
index.add(ids=IDS, texts=CORPUS, vectors=vectors)
print(f"Indexed {index.size} products  (index_type={index.index_type})")

BM25S Count Tokens:   0%|          | 0/44072 [00:00<?, ?it/s]

BM25S Compute Scores:   0%|          | 0/44072 [00:00<?, ?it/s]

Indexed 44072 products  (index_type=helix)
CPU times: user 1.44 s, sys: 35.1 ms, total: 1.48 s
Wall time: 613 ms


In [16]:
def show(title, results):
    print(title)
    if not results:
        print("  (no results)")
    for r in results:
        m = id_to_meta[r.id]
        print(f"  id={r.id}  rank={r.rank}  score={r.score:.4f}  |  {id_to_text[r.id][:55]}  [{m['colour']} {m['type']}]")
    print()

QUERY = "comfortable shoes for working out at the gym black"
q_vec = model.encode([QUERY], normalize_embeddings=True, show_progress_bar=True).astype(np.float32)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

In [17]:
%%time
print(f"Query: {QUERY!r}\n")
show("Semantic-only:", index.search(query_vector=q_vec, k=5))
show("Hybrid:", index.search(query_text=QUERY, query_vector=q_vec, k=5))

Query: 'comfortable shoes for working out at the gym black'

Semantic-only:
  id=prod_42951  rank=0  score=0.3333  |  ADIDAS Women Adi Trainer Black Sports Shoes  [Black Sports Shoes]
  id=prod_31673  rank=1  score=0.2500  |  Nike Women Black Shoes  [Black Casual Shoes]
  id=prod_20319  rank=2  score=0.2000  |  Nike Men Black Shoes  [Black Casual Shoes]
  id=prod_20196  rank=3  score=0.1667  |  Nike Men Black Shoes  [Black Casual Shoes]
  id=prod_4042  rank=4  score=0.1429  |  Nike Men Black Shoes  [Black Casual Shoes]

Hybrid:
  id=prod_36485  rank=0  score=0.4242  |  Fastrack Men Black Gym Bag  [Black Duffel Bag]
  id=prod_43237  rank=1  score=0.4103  |  ADIDAS Men's Gym Polo Blue Black T-Shirt  [Blue Tshirts]
  id=prod_38509  rank=2  score=0.3500  |  Fastrack Men Black Gym Bag  [Black Duffel Bag]
  id=prod_35211  rank=3  score=0.3250  |  Nike Men Gym Grey T-shirt  [Grey Tshirts]
  id=prod_23096  rank=4  score=0.3250  |  Facit Black Comfort Briefs  [Black Briefs]

CPU times: user 110